# Training a new tokenizer from an old one

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!apt install git-lfs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 904.9 kB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 3,544 kB of archives.
After this operation, 10.5 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 git-lfs amd64 3.0.2-1ubuntu0.3 [3,544 kB]
Fetched 3,544 kB in 0s (9,969 kB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.0.2-1ubuntu0.3_amd64.deb ...
Unpacking git-lfs (3.0.2-1ubuntu0.3) ...
Setting up git-lfs (3.0.2-1ubuntu0.3) ...
Processing triggers for man-db (2.10.2-1) ...


You will need to setup git, adapt your email and name in the following cell.

In [ ]:
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [2]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
from datasets import load_dataset

# This can take a few minutes to load, so grab a coffee or tea while you wait!
raw_datasets = load_dataset("code-search-net/code_search_net", "python")

README.md:   0%|          | 0.00/14.1k [00:00<?, ?B/s]

python/train-00000-of-00001.parquet:   0%|          | 0.00/522M [00:00<?, ?B/s]

python/test-00000-of-00001.parquet:   0%|          | 0.00/28.7M [00:00<?, ?B/s]

python/validation-00000-of-00001.parquet:   0%|          | 0.00/30.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

In [7]:
raw_datasets["train"]

Dataset({
    features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
    num_rows: 412178
})

In [8]:
print(raw_datasets["train"][123456]["whole_func_string"])

def oauth_token_create(self, data, **kwargs):
        "https://developer.zendesk.com/rest_api/docs/core/oauth_tokens#create-token"
        api_path = "/api/v2/oauth/tokens.json"
        return self.call(api_path, method="POST", data=data, **kwargs)


In [ ]:
# Don't uncomment the following line unless your dataset is small!
# training_corpus = [raw_datasets["train"][i: i + 1000]["whole_func_string"] for i in range(0, len(raw_datasets["train"]), 1000)]

In [9]:
training_corpus = (
    raw_datasets["train"][i : i + 1000]["whole_func_string"]
    for i in range(0, len(raw_datasets["train"]), 1000)
)

In [10]:
gen = (i for i in range(10))
print(list(gen))
print(list(gen))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[]


In [11]:
def get_training_corpus():
    return (
        raw_datasets["train"][i : i + 1000]["whole_func_string"]
        for i in range(0, len(raw_datasets["train"]), 1000)
    )


training_corpus = get_training_corpus()

In [13]:
def get_training_corpus():
    dataset = raw_datasets["train"]
    for start_idx in range(0, len(dataset), 1000):
        samples = dataset[start_idx : start_idx + 1000]
        yield samples["whole_func_string"]

In [14]:
from transformers import AutoTokenizer

old_tokenizer = AutoTokenizer.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [15]:
example = '''def add_numbers(a, b):
    """Add the two numbers `a` and `b`."""
    return a + b'''

tokens = old_tokenizer.tokenize(example)
tokens

['def',
 'Ġadd',
 '_',
 'n',
 'umbers',
 '(',
 'a',
 ',',
 'Ġb',
 '):',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ"""',
 'Add',
 'Ġthe',
 'Ġtwo',
 'Ġnumbers',
 'Ġ`',
 'a',
 '`',
 'Ġand',
 'Ġ`',
 'b',
 '`',
 '."',
 '""',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġreturn',
 'Ġa',
 'Ġ+',
 'Ġb']

In [16]:
tokenizer = old_tokenizer.train_new_from_iterator(training_corpus, 52000)

In [17]:
tokens = tokenizer.tokenize(example)
tokens

['d',
 'e',
 'f',
 'Ġ',
 'a',
 'd',
 'd',
 '_',
 'n',
 'u',
 'm',
 'b',
 'e',
 'r',
 's',
 '(',
 'a',
 ',',
 'Ġ',
 'b',
 ')',
 ':',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 '"',
 '"',
 '"',
 'A',
 'd',
 'd',
 'Ġ',
 't',
 'h',
 'e',
 'Ġ',
 't',
 'w',
 'o',
 'Ġ',
 'n',
 'u',
 'm',
 'b',
 'e',
 'r',
 's',
 'Ġ',
 '`',
 'a',
 '`',
 'Ġ',
 'a',
 'n',
 'd',
 'Ġ',
 '`',
 'b',
 '`',
 '.',
 '"',
 '"',
 '"',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'r',
 'e',
 't',
 'u',
 'r',
 'n',
 'Ġ',
 'a',
 'Ġ',
 '+',
 'Ġ',
 'b']

In [18]:
print(len(tokens))
print(len(old_tokenizer.tokenize(example)))

82
36


In [19]:
example = """class LinearLayer():
    def __init__(self, input_size, output_size):
        self.weight = torch.randn(input_size, output_size)
        self.bias = torch.zeros(output_size)

    def __call__(self, x):
        return x @ self.weights + self.bias
    """
tokenizer.tokenize(example)

['c',
 'l',
 'a',
 's',
 's',
 'Ġ',
 'L',
 'i',
 'n',
 'e',
 'a',
 'r',
 'L',
 'a',
 'y',
 'e',
 'r',
 '(',
 ')',
 ':',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'd',
 'e',
 'f',
 'Ġ',
 '_',
 '_',
 'i',
 'n',
 'i',
 't',
 '_',
 '_',
 '(',
 's',
 'e',
 'l',
 'f',
 ',',
 'Ġ',
 'i',
 'n',
 'p',
 'u',
 't',
 '_',
 's',
 'i',
 'z',
 'e',
 ',',
 'Ġ',
 'o',
 'u',
 't',
 'p',
 'u',
 't',
 '_',
 's',
 'i',
 'z',
 'e',
 ')',
 ':',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 's',
 'e',
 'l',
 'f',
 '.',
 'w',
 'e',
 'i',
 'g',
 'h',
 't',
 'Ġ',
 '=',
 'Ġ',
 't',
 'o',
 'r',
 'c',
 'h',
 '.',
 'r',
 'a',
 'n',
 'd',
 'n',
 '(',
 'i',
 'n',
 'p',
 'u',
 't',
 '_',
 's',
 'i',
 'z',
 'e',
 ',',
 'Ġ',
 'o',
 'u',
 't',
 'p',
 'u',
 't',
 '_',
 's',
 'i',
 'z',
 'e',
 ')',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 's',
 'e',
 'l',
 'f',
 '.',
 'b',
 'i',
 'a',
 's',
 'Ġ',
 '=',
 'Ġ',
 't',
 'o',
 'r',
 'c',
 'h',
 '.',
 'z',
 'e',
 'r',
 'o',
 's',
 '(',
 'o',
 'u',
 't',
 'p',
 'u',
 't'

In [20]:
tokenizer.save_pretrained("code-search-net-tokenizer")

('code-search-net-tokenizer/tokenizer_config.json',
 'code-search-net-tokenizer/tokenizer.json')

In [21]:
from huggingface_hub import notebook_login

notebook_login()

In [22]:
tokenizer.push_to_hub("code-search-net-tokenizer")

CommitInfo(commit_url='https://huggingface.co/sakib078/code-search-net-tokenizer/commit/cecb46fce277d90f79ac083f8a113f816e886d08', commit_message='Upload tokenizer', commit_description='', oid='cecb46fce277d90f79ac083f8a113f816e886d08', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sakib078/code-search-net-tokenizer', endpoint='https://huggingface.co', repo_type='model', repo_id='sakib078/code-search-net-tokenizer'), pr_revision=None, pr_num=None)

In [23]:
# Replace "huggingface-course" below with your actual namespace to use your own tokenizer
tokenizer = AutoTokenizer.from_pretrained("sakib078/code-search-net-tokenizer")

tokenizer_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

In [24]:
sentence = "tokenizer are greate way to split the lagre corpus of data with better accuracy, It also hadles edge cases."

tokenizer.tokenize(sentence)

['t',
 'o',
 'k',
 'e',
 'n',
 'i',
 'z',
 'e',
 'r',
 'Ġ',
 'a',
 'r',
 'e',
 'Ġ',
 'g',
 'r',
 'e',
 'a',
 't',
 'e',
 'Ġ',
 'w',
 'a',
 'y',
 'Ġ',
 't',
 'o',
 'Ġ',
 's',
 'p',
 'l',
 'i',
 't',
 'Ġ',
 't',
 'h',
 'e',
 'Ġ',
 'l',
 'a',
 'g',
 'r',
 'e',
 'Ġ',
 'c',
 'o',
 'r',
 'p',
 'u',
 's',
 'Ġ',
 'o',
 'f',
 'Ġ',
 'd',
 'a',
 't',
 'a',
 'Ġ',
 'w',
 'i',
 't',
 'h',
 'Ġ',
 'b',
 'e',
 't',
 't',
 'e',
 'r',
 'Ġ',
 'a',
 'c',
 'c',
 'u',
 'r',
 'a',
 'c',
 'y',
 ',',
 'Ġ',
 'I',
 't',
 'Ġ',
 'a',
 'l',
 's',
 'o',
 'Ġ',
 'h',
 'a',
 'd',
 'l',
 'e',
 's',
 'Ġ',
 'e',
 'd',
 'g',
 'e',
 'Ġ',
 'c',
 'a',
 's',
 'e',
 's',
 '.']